# Sistem Case-Based Reasoning (CBR)
## Analisis Putusan Pidana – Kejahatan Terhadap Ketertiban Umum

**Mata Kuliah:** Penalaran Komputer — SubCPMK-3  
**Sumber Data:** Direktori Putusan Mahkamah Agung RI  
**Jumlah Dokumen:** 31 putusan  

---

## ⚠️ Sebelum Menjalankan Notebook Ini

Pastikan kamu sudah menjalankan script berikut dari terminal di folder `cbr-ketertiban-umum/`:

```bash
# Tahap 1 – hanya jika belum punya file .txt (sudah ada? skip)
python scripts/01_proses_pdf.py

# Tahap 2 – hanya jika belum punya cases.json (sudah ada? skip)
python scripts/02_representasi.py
```

Setelah itu, pastikan struktur folder seperti ini:
```
cbr-ketertiban-umum/
├── data/
│   ├── raw/
│   │   ├── case_001.txt
│   │   ├── case_002.txt  (dst...)
│   │   └── metadata_raw.json
│   └── processed/
│       └── cases.json      ← WAJIB ADA sebelum lanjut
└── notebooks/
    └── CBR_Ketertiban_Umum.ipynb  ← file ini
```

## Setup & Import Library

In [ ]:
# Install dependencies jika belum ada
# Hapus tanda # di bawah ini lalu jalankan jika pertama kali
# !pip install scikit-learn pandas numpy matplotlib seaborn tqdm

In [ ]:
import re
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# ── Deteksi ROOT_DIR secara otomatis & robust ──────────────────────────────
# Notebook ini ada di: cbr-ketertiban-umum/notebooks/
# ROOT_DIR harusnya : cbr-ketertiban-umum/
import os

# Cara 1: dari lokasi file notebook (VS Code / JupyterLab)
_nb_file = globals().get('__vsc_ipynb_file__', '')
if _nb_file:
    ROOT_DIR = Path(_nb_file).resolve().parent.parent
else:
    # Cara 2: deteksi dari working directory saat ini
    _cwd = Path(os.getcwd()).resolve()
    if _cwd.name == 'notebooks':
        # Jupyter dibuka dari dalam folder notebooks/
        ROOT_DIR = _cwd.parent
    elif (_cwd / 'notebooks').exists() and (_cwd / 'scripts').exists():
        # Jupyter dibuka dari root project
        ROOT_DIR = _cwd
    elif (_cwd.parent / 'notebooks').exists():
        # Satu level di atas
        ROOT_DIR = _cwd.parent
    else:
        ROOT_DIR = _cwd.parent  # fallback

RAW_DIR       = ROOT_DIR / 'data' / 'raw'
PROCESSED_DIR = ROOT_DIR / 'data' / 'processed'
EVAL_DIR      = ROOT_DIR / 'data' / 'eval'
RESULTS_DIR   = ROOT_DIR / 'data' / 'results'
FIGS_DIR      = ROOT_DIR / 'figures'

for d in [EVAL_DIR, RESULTS_DIR, FIGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('✓ Setup selesai')
print(f'  Working dir : {os.getcwd()}')
print(f'  Root project: {ROOT_DIR}')
print(f'  Raw dir     : {RAW_DIR}')
print(f'  Processed   : {PROCESSED_DIR}')
print()

# Cek file wajib
cases_json = PROCESSED_DIR / 'cases.json'
raw_exists  = RAW_DIR.exists() and len(list(RAW_DIR.glob('case_*.txt'))) > 0

if not raw_exists:
    print('⚠️  File .txt tidak ditemukan di data/raw/')
    print('   → Jalankan dulu dari terminal DI FOLDER ROOT PROJECT:')
    print(f'   cd {ROOT_DIR}')
    print('   python scripts/01_proses_pdf.py')
elif not cases_json.exists():
    print('⚠️  cases.json tidak ditemukan di data/processed/')
    print('   → Jalankan dulu dari terminal DI FOLDER ROOT PROJECT:')
    print(f'   cd {ROOT_DIR}')
    print('   python scripts/02_representasi.py')
else:
    n_txt = len(list(RAW_DIR.glob('case_*.txt')))
    print(f'✓ {n_txt} file .txt ditemukan di data/raw/')
    print('✓ cases.json ditemukan → siap lanjut ke cell berikutnya!')


---
## Tahap 1: Membangun Case Base
Data telah diunduh dari [Direktori Putusan MA RI](https://putusan3.mahkamahagung.go.id) sebanyak **31 dokumen** putusan Pidana Kejahatan Terhadap Ketertiban Umum dan diekstrak menggunakan `scripts/01_proses_pdf.py`.

In [ ]:
# Cek hasil Tahap 1
txt_files = sorted(RAW_DIR.glob('case_*.txt'))
meta_path = RAW_DIR / 'metadata_raw.json'

print(f'Jumlah file .txt : {len(txt_files)}')

if meta_path.exists():
    with open(meta_path, encoding='utf-8') as f:
        metadata_raw = json.load(f)
    meta_df = pd.DataFrame(metadata_raw)
    print(f'Entri metadata   : {len(meta_df)}')
    print(f"\nStatus ekstraksi:\n{meta_df['status'].value_counts().to_string()}")
    display(meta_df[['case_id','file_asli','word_count','status']].head(5))
else:
    print('metadata_raw.json belum ada — jalankan scripts/01_proses_pdf.py')

In [ ]:
# Contoh teks hasil ekstraksi
if txt_files:
    sample = txt_files[0].read_text(encoding='utf-8', errors='ignore')
    print(f'File  : {txt_files[0].name}')
    print(f'Kata  : {len(sample.split()):,}')
    print(f'\nCuplikan 500 karakter pertama:')
    print('-'*60)
    print(sample[:500])

---
## Tahap 2: Case Representation
Metadata diekstrak menggunakan `scripts/02_representasi.py` dan disimpan ke `data/processed/cases.json`.

In [ ]:
# Load cases.json
with open(PROCESSED_DIR / 'cases.json', encoding='utf-8') as f:
    cases_data = json.load(f)

df = pd.DataFrame(cases_data)
print(f'Total kasus  : {len(df)}')
print(f'Kolom        : {list(df.columns)}')
print(f'text_full OK : {(df["text_full"].str.len() > 100).sum()}/{len(df)}')
print(f'\nDistribusi label:')
print(df['label'].value_counts().to_string())

display(df[['case_id','no_perkara','tanggal','pengadilan','terdakwa',
            'pasal','vonis','label','word_count']].head(10))

In [ ]:
# Simpan cases.csv (tanpa text_full)
df.drop(columns=['text_full'], errors='ignore').to_csv(
    PROCESSED_DIR / 'cases.csv', index=False, encoding='utf-8-sig')
print('✓ cases.csv disimpan')

In [ ]:
# Visualisasi distribusi data
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Label
lc = df['label'].value_counts()
axes[0].bar(lc.index, lc.values,
            color=['#4C72B0','#DD8452','#55A868','#C44E52','#9B59B6'][:len(lc)],
            alpha=0.85)
axes[0].set_title('Distribusi Label Putusan', fontweight='bold')
axes[0].set_xlabel('Label'); axes[0].set_ylabel('Jumlah')
axes[0].tick_params(axis='x', rotation=20)
for i, v in enumerate(lc.values):
    axes[0].text(i, v+0.1, str(v), ha='center', fontweight='bold')

# 2. Word count
axes[1].hist(df['word_count'].dropna(), bins=15,
             color='#4C72B0', edgecolor='white', alpha=0.85)
axes[1].set_title('Distribusi Jumlah Kata', fontweight='bold')
axes[1].set_xlabel('Jumlah Kata'); axes[1].set_ylabel('Frekuensi')
axes[1].axvline(df['word_count'].median(), color='red', linestyle='--',
                label=f'Median: {int(df["word_count"].median())}')
axes[1].legend()

# 3. Pengadilan
pc = df['pengadilan'].value_counts().head(7)
axes[2].barh(pc.index, pc.values, color='#55A868', alpha=0.85)
axes[2].set_title('Top 7 Pengadilan', fontweight='bold')
axes[2].set_xlabel('Jumlah Kasus')

plt.suptitle('Eksplorasi Data – Putusan Ketertiban Umum (31 Dokumen)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'data_exploration.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Visualisasi disimpan di figures/data_exploration.png')

---
## Tahap 3: Case Retrieval

**Dua pendekatan:**
| Model | Vectorizer | Split |
|-------|-----------|-------|
| SVM (LinearSVC) | TF-IDF | 80:20 |
| Naive Bayes (MultinomialNB) | TF-IDF | 80:20 |

In [ ]:
# Persiapan: gunakan hanya kasus berlabel
df_labeled = df[
    (df['label'] != 'tidak_diketahui') &
    (df['text_full'].str.len() > 100)
].copy().reset_index(drop=True)

print(f'Kasus berlabel   : {len(df_labeled)}')
print(f'Tidak diketahui  : {len(df) - len(df_labeled)}')
print(f'\nDistribusi label training:')
print(df_labeled['label'].value_counts().to_string())

texts  = df_labeled['text_full'].tolist()
labels = df_labeled['label'].tolist()
ids    = df_labeled['case_id'].tolist()

le = LabelEncoder()
y  = le.fit_transform(labels)
print(f'\nKelas: {le.classes_}')

In [ ]:
# Split 80:20
try:
    train_idx, test_idx = train_test_split(
        range(len(texts)), test_size=0.2, random_state=42, stratify=y)
except ValueError:
    train_idx, test_idx = train_test_split(
        range(len(texts)), test_size=0.2, random_state=42)

train_idx, test_idx = list(train_idx), list(test_idx)
print(f'Train : {len(train_idx)} kasus')
print(f'Test  : {len(test_idx)} kasus')

with open(EVAL_DIR / 'split_info.json', 'w') as f:
    json.dump({'train_idx': train_idx, 'test_idx': test_idx}, f)
print('✓ split_info.json disimpan')

In [ ]:
# TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=1,
    strip_accents='unicode'
)

X_all   = tfidf.fit_transform(texts)
X_train = X_all[train_idx]
X_test  = X_all[test_idx]
y_train = y[train_idx]
y_test  = y[test_idx]

print(f'Vocabulary size : {len(tfidf.vocabulary_):,}')
print(f'Train matrix    : {X_train.shape}')
print(f'Test matrix     : {X_test.shape}')

In [ ]:
# Case base = semua 31 dokumen
all_texts  = df['text_full'].fillna('').tolist()
all_ids    = df['case_id'].tolist()
X_corpus   = tfidf.transform(all_texts)

# Fungsi retrieve()
def retrieve(query: str, k: int = 5) -> list:
    """
    Kembalikan top-k case_id paling mirip dengan query.
    1) Pre-process query
    2) Hitung vektor query (TF-IDF)
    3) Hitung cosine-similarity dengan semua case vectors
    4) Kembalikan top-k case_id
    """
    q_vec = tfidf.transform([query])
    sims  = cosine_similarity(q_vec, X_corpus).flatten()
    top_k = np.argsort(sims)[::-1][:k]
    return [(all_ids[i], round(float(sims[i]), 4)) for i in top_k]

# Quick test
q_test = 'terdakwa bersama-sama melakukan kekerasan dengan tenaga bersama terhadap orang lain'
print(f'Query: "{q_test}"')
print('\nTop-5 kasus paling mirip:')
for rank, (cid, score) in enumerate(retrieve(q_test), 1):
    row = df[df['case_id']==cid].iloc[0]
    print(f'  {rank}. {cid} | sim={score} | label={row["label"]} | {row["no_perkara"]}')

In [ ]:
# Training SVM
svm_model = LinearSVC(C=1.0, max_iter=5000, random_state=42)
svm_model.fit(X_train, y_train)
print('✓ TF-IDF + SVM dilatih')

# Training Naive Bayes
nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train, y_train)
print('✓ TF-IDF + Naive Bayes dilatih')

# Simpan model
with open(PROCESSED_DIR / 'models.pkl', 'wb') as f:
    pickle.dump({'tfidf': tfidf, 'svm': svm_model, 'nb': nb_model, 'le': le}, f)
print('✓ Model disimpan: data/processed/models.pkl')

In [ ]:
# Simpan queries.json
eval_queries = [
    {'query_id': f'q_{ids[i]}', 'query_text': texts[i][:300],
     'ground_truth': ids[i], 'true_label': labels[i]}
    for i in test_idx
]
with open(EVAL_DIR / 'queries.json', 'w', encoding='utf-8') as f:
    json.dump(eval_queries, f, ensure_ascii=False, indent=2)
print(f'✓ queries.json disimpan ({len(eval_queries)} query)')

---
## Tahap 4: Case Solution Reuse
Retrieve top-5 kasus mirip → weighted similarity vote → prediksi label & amar putusan.

In [ ]:
# Bangun kamus solusi dari semua kasus
case_solutions = {
    row['case_id']: {
        'amar_putusan': str(row.get('amar_putusan', '')),
        'vonis'       : str(row.get('vonis', '')),
        'label'       : str(row.get('label', '')),
        'no_perkara'  : str(row.get('no_perkara', '')),
        'pasal'       : str(row.get('pasal', '')),
    }
    for _, row in df.iterrows()
}
print(f'✓ Solusi dimuat untuk {len(case_solutions)} kasus')

In [ ]:
# Fungsi predict_outcome()
def predict_outcome(query: str, model: str = 'svm', k: int = 5) -> dict:
    top_k = retrieve(query, k=k)

    # Weighted similarity vote
    weights = {}
    amars   = {}
    for cid, score in top_k:
        sol = case_solutions.get(cid, {})
        lbl = sol.get('label', 'tidak_diketahui')
        weights[lbl] = weights.get(lbl, 0) + score
        if lbl not in amars:
            amars[lbl] = sol.get('amar_putusan', '')[:200]

    best_label = max(weights, key=weights.__getitem__)

    # Prediksi via classifier
    q_vec    = tfidf.transform([query])
    clf      = svm_model if model == 'svm' else nb_model
    clf_pred = le.inverse_transform(clf.predict(q_vec))[0]

    return {
        'predicted_label': clf_pred,
        'weighted_label' : best_label,
        'predicted_amar' : amars.get(best_label, ''),
        'top_k'          : top_k,
        'weight_dist'    : {k: round(v, 4) for k, v in weights.items()},
        'top_k_details'  : [
            {'case_id': cid, 'similarity': score,
             'label': case_solutions.get(cid, {}).get('label', ''),
             'no_perkara': case_solutions.get(cid, {}).get('no_perkara', ''),
             'vonis': case_solutions.get(cid, {}).get('vonis', '')}
            for cid, score in top_k
        ]
    }

In [ ]:
# Demo: 5 kasus baru
DEMO_QUERIES = [
    {'id':'demo_001', 'true':'bersalah_ringan',
     'query':'terdakwa bersama-sama melakukan kekerasan terang-terangan di muka umum menggunakan senjata tajam melukai korban hingga luka berat'},
    {'id':'demo_002', 'true':'bersalah_ringan',
     'query':'terdakwa memasuki rumah orang lain secara paksa tanpa izin menggunakan kekerasan dan mengancam penghuni rumah'},
    {'id':'demo_003', 'true':'bersalah_berat',
     'query':'terdakwa terbukti melakukan penghasutan kepada massa untuk melakukan kekerasan terhadap kelompok tertentu secara sistematis'},
    {'id':'demo_004', 'true':'bebas',
     'query':'terdakwa tidak terbukti melakukan tindak pidana kekerasan berdasarkan alat bukti yang ada sehingga dibebaskan dari segala dakwaan'},
    {'id':'demo_005', 'true':'bersalah_berat',
     'query':'terdakwa bersama kelompok merusak fasilitas umum dan menyerang petugas keamanan saat demonstrasi menyebabkan kerugian besar'},
]

print('='*65)
print('DEMO PREDIKSI 5 KASUS BARU')
print('='*65)

demo_rows = []
for d in DEMO_QUERIES:
    res_svm = predict_outcome(d['query'], model='svm')
    res_nb  = predict_outcome(d['query'], model='nb')
    match_s = '✓' if res_svm['predicted_label'] == d['true'] else '✗'
    match_n = '✓' if res_nb['predicted_label']  == d['true'] else '✗'

    print(f"\n[{d['id']}] {d['query'][:70]}...")
    print(f"  True label   : {d['true']}")
    print(f"  SVM prediksi : {res_svm['predicted_label']} {match_s}")
    print(f"  NB  prediksi : {res_nb['predicted_label']}  {match_n}")
    print(f"  Top-3 mirip  :")
    for i, det in enumerate(res_svm['top_k_details'][:3], 1):
        print(f"    {i}. {det['case_id']} | sim={det['similarity']} | {det['label']} | vonis: {det['vonis'] or '-'}")

    demo_rows.append({
        'query_id'      : d['id'],
        'query_text'    : d['query'],
        'true_label'    : d['true'],
        'predicted_svm' : res_svm['predicted_label'],
        'predicted_nb'  : res_nb['predicted_label'],
        'predicted_amar': res_svm['predicted_amar'][:100],
        'top_5_case_ids': '|'.join([x[0] for x in res_svm['top_k']]),
        'correct_svm'   : res_svm['predicted_label'] == d['true'],
        'correct_nb'    : res_nb['predicted_label']  == d['true'],
    })

demo_df = pd.DataFrame(demo_rows)
demo_df.to_csv(RESULTS_DIR / 'predictions_demo.csv', index=False, encoding='utf-8-sig')
print(f'\n✓ predictions_demo.csv disimpan')

---
## Tahap 5: Model Evaluation
**Metrik:** Accuracy, Precision, Recall, F1-Score (weighted)

In [ ]:
# Prediksi pada test set
pred_svm = svm_model.predict(X_test)
pred_nb  = nb_model.predict(X_test)

true_lbl     = le.inverse_transform(y_test)
pred_svm_lbl = le.inverse_transform(pred_svm)
pred_nb_lbl  = le.inverse_transform(pred_nb)

# Top-5 retrieval accuracy
hit = sum(1 for i in test_idx
          if ids[i] in [x[0] for x in retrieve(texts[i], k=5)])
topk_acc = round(hit / len(test_idx), 4)

def hitung_metrik(true, pred, name):
    return {
        'model'    : name,
        'accuracy' : round(accuracy_score(true, pred), 4),
        'precision': round(precision_score(true, pred, average='weighted', zero_division=0), 4),
        'recall'   : round(recall_score(true, pred, average='weighted', zero_division=0), 4),
        'f1_score' : round(f1_score(true, pred, average='weighted', zero_division=0), 4),
        'top_5_retrieval_accuracy': topk_acc,
        'n_test'   : len(true),
    }

metrics_df = pd.DataFrame([
    hitung_metrik(true_lbl, pred_svm_lbl, 'TF-IDF + SVM'),
    hitung_metrik(true_lbl, pred_nb_lbl,  'TF-IDF + Naive Bayes'),
])
metrics_df.to_csv(EVAL_DIR / 'retrieval_metrics.csv', index=False, encoding='utf-8-sig')

print('='*60)
print('TABEL METRIK PERBANDINGAN MODEL')
print('='*60)
display(metrics_df.set_index('model'))

In [ ]:
# Classification Report
print('Classification Report – TF-IDF + SVM:')
print(classification_report(true_lbl, pred_svm_lbl, zero_division=0))
print('Classification Report – TF-IDF + Naive Bayes:')
print(classification_report(true_lbl, pred_nb_lbl,  zero_division=0))

In [ ]:
# Bar Chart Perbandingan
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, col, title, color in zip(
    axes,
    ['accuracy','precision','recall','f1_score'],
    ['Accuracy','Precision','Recall','F1-Score'],
    ['#4C72B0','#DD8452','#55A868','#C44E52']
):
    bars = ax.bar(metrics_df['model'], metrics_df[col],
                  color=color, alpha=0.85, width=0.4)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score')
    ax.tick_params(axis='x', rotation=15)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2., h+0.02,
                f'{h:.4f}', ha='center', fontsize=9)

plt.suptitle('Perbandingan Performa Model CBR — Ketertiban Umum',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ figures/metrics_comparison.png disimpan')

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
lbl_unik = sorted(set(true_lbl) | set(pred_svm_lbl) | set(pred_nb_lbl))

for ax, pred, title in zip(
    axes,
    [pred_svm_lbl, pred_nb_lbl],
    ['TF-IDF + SVM', 'TF-IDF + Naive Bayes']
):
    cm = confusion_matrix(true_lbl, pred, labels=lbl_unik)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=lbl_unik, yticklabels=lbl_unik, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix\n{title}', fontweight='bold')
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig(FIGS_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ figures/confusion_matrix.png disimpan')

In [ ]:
# Error Analysis
errors = []
for i, ti in enumerate(test_idx):
    if true_lbl[i] != pred_svm_lbl[i]:
        row = df_labeled.iloc[ti]
        errors.append({
            'case_id'   : ids[ti],
            'no_perkara': row['no_perkara'],
            'true_label': true_lbl[i],
            'pred_svm'  : pred_svm_lbl[i],
            'pred_nb'   : pred_nb_lbl[i],
            'word_count': row.get('word_count', 0),
            'vonis'     : row.get('vonis', ''),
        })

err_df = pd.DataFrame(errors)
err_df.to_csv(EVAL_DIR / 'error_analysis.csv', index=False, encoding='utf-8-sig')

print('='*60)
print('ERROR ANALYSIS – TF-IDF + SVM')
print('='*60)
if len(err_df) == 0:
    print('Tidak ada error pada test set!')
else:
    print(f'Total error: {len(err_df)}/{len(test_idx)} kasus')
    display(err_df)
    print('\nAnalisis penyebab error:')
    print('  - Data test terlalu sedikit (hanya 4 kasus) → hasil kurang representatif')
    print('  - Label tidak_diketahui (12 kasus) tidak digunakan untuk training')
    print('  - Rekomendasi: tambah data minimal 50+ dokumen')
print('✓ error_analysis.csv disimpan')

In [ ]:
# Simpan prediction_metrics.csv
pred_rows = []
for i, ti in enumerate(test_idx):
    res = predict_outcome(texts[ti], model='svm')
    pred_rows.append({
        'query_id'           : f'q_{ids[ti]}',
        'no_perkara'         : df_labeled.iloc[ti]['no_perkara'],
        'true_label'         : true_lbl[i],
        'predicted_label_svm': pred_svm_lbl[i],
        'predicted_label_nb' : pred_nb_lbl[i],
        'predicted_amar'     : res['predicted_amar'][:100],
        'top_5_case_ids'     : '|'.join([x[0] for x in res['top_k']]),
        'correct_svm'        : true_lbl[i] == pred_svm_lbl[i],
        'correct_nb'         : true_lbl[i] == pred_nb_lbl[i],
    })

pd.DataFrame(pred_rows).to_csv(
    EVAL_DIR / 'prediction_metrics.csv', index=False, encoding='utf-8-sig')
print('✓ prediction_metrics.csv disimpan')

---
## Ringkasan Hasil & Rekomendasi

In [ ]:
best = metrics_df.loc[metrics_df['f1_score'].idxmax(), 'model']

print('='*60)
print('RINGKASAN HASIL')
print('='*60)
print(f'Total dokumen        : 31 putusan')
print(f'Kasus berlabel       : {len(df_labeled)}')
print(f'Split train/test     : {len(train_idx)}/{len(test_idx)} (80:20)')
print()
for _, row in metrics_df.iterrows():
    print(f"{row['model']:25s} | Acc={row['accuracy']:.4f} "
          f"| F1={row['f1_score']:.4f} "
          f"| Top-5={row['top_5_retrieval_accuracy']:.4f}")
print(f'\nModel terbaik (F1)   : {best}')
print()
print('Rekomendasi Perbaikan:')
print('  1. Tambah data minimal 50+ dokumen untuk hasil lebih robust')
print('  2. Perbaiki regex vonis agar label tidak_diketahui berkurang')
print('  3. Coba IndoBERT untuk pemahaman konteks bahasa Indonesia')
print('  4. Gunakan k-fold cross validation untuk evaluasi lebih stabil')

In [ ]:
# Cek semua output yang dihasilkan
print('Output yang dihasilkan:')
for folder in [PROCESSED_DIR, EVAL_DIR, RESULTS_DIR, FIGS_DIR]:
    files = sorted(folder.glob('*'))
    if files:
        print(f'\n{folder.relative_to(ROOT_DIR)}/')
        for f in files:
            print(f'  ├── {f.name} ({f.stat().st_size/1024:.1f} KB)')